# Experiment 7: Data Source Evaluation

**Goal:** Evaluate additional data sources (Wikidata, DBpedia, Open Opus) alongside our existing ones (MusicBrainz, Discogs, Wikipedia) to make an informed, documented selection.

**Test artists across genres:**
- Classical: Ludwig van Beethoven
- Jazz: Miles Davis
- Rock: David Bowie
- Brazilian: Tom Jobim (Antônio Carlos Jobim)
- African: Fela Kuti

**Sources to evaluate:**
1. Wikidata SPARQL (structured, RDF)
2. DBpedia (structured, RDF)
3. Open Opus (structured, JSON — classical only)
4. MusicBrainz (already tested — baseline comparison)
5. Discogs (already tested — baseline comparison)

## Setup

In [1]:
import requests
import json
import time
from SPARQLWrapper import SPARQLWrapper, JSON

HEADERS = {"User-Agent": "KE-CW2-MusicHistory/0.1 (lucas@example.com)"}

def pp(data):
    print(json.dumps(data, indent=2, default=str))

# Test artists with Wikidata IDs for direct lookup
TEST_ARTISTS = {
    "Beethoven":    {"genre": "Classical",  "wikidata": "Q255",   "mb": "1f9df192-a621-4f54-8850-2c5373b7eac9"},
    "Miles Davis":  {"genre": "Jazz",       "wikidata": "Q93341", "mb": "561d854a-6a28-4aa7-8c99-323e6ce46c2a"},
    "David Bowie":  {"genre": "Rock",       "wikidata": "Q5383",  "mb": "5441c29d-3602-4898-b1a1-b77fa23b8e50"},
    "Tom Jobim":    {"genre": "Brazilian",  "wikidata": "Q202325","mb": "38bfb8dd-1a5c-4f09-946c-bb1de0520cf7"},
    "Fela Kuti":    {"genre": "African",    "wikidata": "Q207082","mb": "01234d68-fae1-49a6-8d7e-2212e tried8"},
}

print("Test artists loaded.")

Test artists loaded.


## 1. Wikidata SPARQL

Query Wikidata directly for structured music data. Already in RDF — no mapping needed.

In [2]:
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.addCustomHttpHeader("User-Agent", "KE-CW2-MusicHistory/0.1 (lucas@example.com)")

def wikidata_query(query_str):
    sparql.setQuery(query_str)
    sparql.setReturnFormat(JSON)
    return sparql.query().convert()

print("Wikidata SPARQL client ready.")

Wikidata SPARQL client ready.


In [3]:
# Query: Get comprehensive info for all test artists at once
# Properties: genre (P136), instrument (P1303), award (P166), 
#             record label (P264), influenced by (P737), country (P27),
#             birth date (P569), death date (P570), birth place (P19)

wikidata_ids = [info["wikidata"] for info in TEST_ARTISTS.values()]
values_clause = " ".join([f"wd:{qid}" for qid in wikidata_ids])

query = f"""
SELECT ?artist ?artistLabel ?propLabel ?valueLabel WHERE {{
  VALUES ?artist {{ {values_clause} }}
  VALUES ?prop {{ 
    wdt:P136   # genre
    wdt:P1303  # instrument
    wdt:P166   # award received
    wdt:P264   # record label
    wdt:P737   # influenced by
    wdt:P27    # country of citizenship
    wdt:P569   # date of birth
    wdt:P570   # date of death
    wdt:P19    # place of birth
    wdt:P106   # occupation
  }}
  ?artist ?prop ?value .
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
ORDER BY ?artistLabel ?propLabel
"""

results = wikidata_query(query)

# Group by artist
from collections import defaultdict
wd_data = defaultdict(lambda: defaultdict(list))

for r in results["results"]["bindings"]:
    artist = r["artistLabel"]["value"]
    prop = r["propLabel"]["value"]
    value = r["valueLabel"]["value"]
    wd_data[artist][prop].append(value)

for artist, props in sorted(wd_data.items()):
    print(f"\n{'='*60}")
    print(f"{artist}")
    print(f"{'='*60}")
    for prop, values in sorted(props.items()):
        vals = ", ".join(values[:5])
        extra = f" (+{len(values)-5} more)" if len(values) > 5 else ""
        print(f"  {prop:25s}: {vals}{extra}")


David Bowie
  http://www.wikidata.org/prop/direct/P106: actor, composer, singer, record producer, pianist (+17 more)
  http://www.wikidata.org/prop/direct/P1303: piano, guitar, saxophone, keyboard instrument, drum kit (+3 more)
  http://www.wikidata.org/prop/direct/P136: rapping, electronic music, alternative rock, rock music, pop music (+13 more)
  http://www.wikidata.org/prop/direct/P166: Grammy Hall of Fame, Great Britons, Rock and Roll Hall of Fame, Ordre des Arts et des Lettres, Grammy Lifetime Achievement Award (+8 more)
  http://www.wikidata.org/prop/direct/P19: Brixton
  http://www.wikidata.org/prop/direct/P264: Mercury Records, Columbia Records, EMI, RCA Records, Virgin Records (+9 more)
  http://www.wikidata.org/prop/direct/P27: United Kingdom
  http://www.wikidata.org/prop/direct/P569: 1947-01-08T00:00:00Z
  http://www.wikidata.org/prop/direct/P570: 2016-01-10T00:00:00Z
  http://www.wikidata.org/prop/direct/P737: Bob Dylan, The Beatles, Jacques Brel, Pink Floyd, Andy Warhol

In [4]:
# Query: Get notable works for each artist
query_works = f"""
SELECT ?artist ?artistLabel ?workLabel ?workTypeLabel ?date WHERE {{
  VALUES ?artist {{ {values_clause} }}
  ?work wdt:P175 ?artist .     # performer
  OPTIONAL {{ ?work wdt:P31 ?workType . }}  # instance of
  OPTIONAL {{ ?work wdt:P577 ?date . }}     # publication date
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
ORDER BY ?artistLabel ?date
LIMIT 50
"""

works_results = wikidata_query(query_works)

print("Notable works found in Wikidata:\n")
current = None
for r in works_results["results"]["bindings"]:
    artist = r["artistLabel"]["value"]
    work = r["workLabel"]["value"]
    wtype = r.get("workTypeLabel", {}).get("value", "?")
    date = r.get("date", {}).get("value", "?")[:10]
    if artist != current:
        current = artist
        print(f"\n  {artist}:")
    print(f"    - {work} ({wtype}, {date})")

Notable works found in Wikidata:


  David Bowie:
    - Queen Bitch (musical work/composition, ?)
    - Santa Monica '72 (album, ?)
    - Ziggy Stardust Tour (concert tour, ?)
    - Ziggy Stardust (alter ego, ?)
    - Ziggy Stardust (persona, ?)
    - Ziggy Stardust (music video character, ?)
    - Ziggy Stardust (fictional humanoid extraterrestrial, ?)
    - VH1 Storytellers (album, ?)
    - Isolar – 1976 Tour (concert tour, ?)
    - Quicksand (musical work/composition, ?)
    - Seven Years in Tibet (musical work/composition, ?)
    - Diamond Dogs Tour (concert tour, ?)
    - Outside Summer Festivals Tour (concert tour, ?)
    - Outside Tour (concert tour, ?)
    - BBC Sessions 1969–1972 (album, ?)
    - Bring Me the Disco King (musical work/composition, ?)
    - Earthling Tour (concert tour, ?)
    - Fall Dog Bombs the Moon (musical work/composition, ?)
    - Glass Spider Tour (concert tour, ?)
    - Heathen Tour (concert tour, ?)
    - Here Today and Gone Tomorrow (musical work/comp

In [5]:
# Query: Get compositions (for classical — using P86 composer)
query_compositions = """
SELECT ?compositionLabel ?genreLabel ?date ?keyLabel WHERE {
  ?composition wdt:P86 wd:Q255 .  # composer = Beethoven
  OPTIONAL { ?composition wdt:P136 ?genre . }    # genre
  OPTIONAL { ?composition wdt:P577 ?date . }     # publication date
  OPTIONAL { ?composition wdt:P826 ?key . }      # musical key
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
ORDER BY ?date
LIMIT 20
"""

comp_results = wikidata_query(query_compositions)

print("Beethoven compositions in Wikidata:\n")
for r in comp_results["results"]["bindings"]:
    title = r["compositionLabel"]["value"]
    genre = r.get("genreLabel", {}).get("value", "?")
    date = r.get("date", {}).get("value", "?")[:10]
    key = r.get("keyLabel", {}).get("value", "?")
    print(f"  {title:50s} | Genre: {genre:20s} | Date: {date} | Key: {key}")

Beethoven compositions in Wikidata:

  Q97192827                                          | Genre: ?                    | Date: ? | Key: ?
  Q104897606                                         | Genre: documentary film     | Date: ? | Key: ?
  Q105197451                                         | Genre: ?                    | Date: ? | Key: ?
  Gustav Mahler's orchestration of Beethoven's Symphony No. 9 | Genre: classical music      | Date: ? | Key: ?
  Adagio Hammerklavier                               | Genre: ballet               | Date: ? | Key: ?
  Q114207503                                         | Genre: ?                    | Date: ? | Key: ?
  Sehnsucht, WoO 134                                 | Genre: ?                    | Date: ? | Key: ?
  Beethoven in Havana                                | Genre: Cuban rumba          | Date: ? | Key: ?
  Q116975000                                         | Genre: ?                    | Date: ? | Key: ?
  Månaderna                         

## 2. DBpedia

Wikipedia data already in RDF. Query via SPARQL.

In [6]:
dbpedia = SPARQLWrapper("https://dbpedia.org/sparql")
dbpedia.addCustomHttpHeader("User-Agent", "KE-CW2-MusicHistory/0.1 (lucas@example.com)")

def dbpedia_query(query_str):
    dbpedia.setQuery(query_str)
    dbpedia.setReturnFormat(JSON)
    return dbpedia.query().convert()

print("DBpedia SPARQL client ready.")

DBpedia SPARQL client ready.


In [7]:
# Query DBpedia for David Bowie
dbp_query = """
SELECT ?prop ?value WHERE {
  <http://dbpedia.org/resource/David_Bowie> ?prop ?value .
  FILTER(LANG(?value) = 'en' || !isLiteral(?value))
}
LIMIT 100
"""

dbp_results = dbpedia_query(dbp_query)

print("DBpedia properties for David Bowie:\n")
for r in dbp_results["results"]["bindings"]:
    prop = r["prop"]["value"].split("/")[-1].split("#")[-1]
    value = r["value"]["value"]
    if len(value) > 100:
        value = value[:100] + "..."
    print(f"  {prop:40s}: {value}")

DBpedia properties for David Bowie:

  label                                   : David Bowie
  deathPlace                              : New York City, US
  birthPlace                              : London, England
  type                                    : http://www.w3.org/2002/07/owl#Thing
  type                                    : http://xmlns.com/foaf/0.1/Person
  type                                    : http://dbpedia.org/ontology/Person
  type                                    : http://dbpedia.org/ontology/Person
  type                                    : http://dbpedia.org/ontology/Person
  type                                    : http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#NaturalPerson
  type                                    : http://www.wikidata.org/entity/Q19088
  type                                    : http://www.wikidata.org/entity/Q215627
  type                                    : http://www.wikidata.org/entity/Q5
  type                              

In [8]:
# Query DBpedia for Fela Kuti (test African artist coverage)
dbp_fela = """
SELECT ?prop ?value WHERE {
  <http://dbpedia.org/resource/Fela_Kuti> ?prop ?value .
  FILTER(LANG(?value) = 'en' || !isLiteral(?value))
}
LIMIT 100
"""

fela_results = dbpedia_query(dbp_fela)

print("DBpedia properties for Fela Kuti:\n")
for r in fela_results["results"]["bindings"]:
    prop = r["prop"]["value"].split("/")[-1].split("#")[-1]
    value = r["value"]["value"]
    if len(value) > 100:
        value = value[:100] + "..."
    print(f"  {prop:40s}: {value}")

DBpedia properties for Fela Kuti:

  name                                    : Fela Kuti
  label                                   : Fela Kuti
  deathPlace                              : Lagos, Lagos State, Nigeria
  birthPlace                              : Abeokuta, British Nigeria
  description                             : Nigerian musician and activist (1938–1997)
  name                                    : Fela Kuti
  id                                      : mn0000138833
  birthName                               : Olufela Olusegun Oludotun Ransome-Kuti
  b                                       : no
  n                                       : no
  occupation                              : 
  occupation                              : Musician
  occupation                              : activist
  occupation                              : bandleader
  quote                                   : "Imagine Che Guevara and Bob Marley rolled into one person and you get a sense of Nigerian

In [9]:
# Query DBpedia for Tom Jobim (test Brazilian artist coverage)
dbp_jobim = """
SELECT ?prop ?value WHERE {
  <http://dbpedia.org/resource/Antônio_Carlos_Jobim> ?prop ?value .
  FILTER(LANG(?value) = 'en' || !isLiteral(?value))
}
LIMIT 100
"""

jobim_results = dbpedia_query(dbp_jobim)

print("DBpedia properties for Tom Jobim:\n")
for r in jobim_results["results"]["bindings"]:
    prop = r["prop"]["value"].split("/")[-1].split("#")[-1]
    value = r["value"]["value"]
    if len(value) > 100:
        value = value[:100] + "..."
    print(f"  {prop:40s}: {value}")

DBpedia properties for Tom Jobim:

  name                                    : Tom Jobim
  label                                   : Antônio Carlos Jobim
  deathPlace                              : New York City, U.S.
  birthPlace                              : Rio de Janeiro, Brazil
  givenName                               : Antônio Carlos Brasileiro de Almeida Jobim
  description                             : Brazilian songwriter, composer, arranger, singer, and pianist/guitarist (1927–1994)
  name                                    : Tom Jobim
  genre                                   : Bossa nova
  occupation                              : 
  occupation                              : Musician
  occupation                              : composer
  occupation                              : singer
  occupation                              : songwriter
  birthName                               : Antônio Carlos Brasileiro de Almeida Jobim
  alias                                   : ,
 

## 3. Open Opus (Classical Music API)

Free API specifically for classical music. No auth needed.

In [10]:
# Search for Beethoven
oo_resp = requests.get(
    "https://api.openopus.org/composer/list/search/beethoven.json",
    headers=HEADERS
)

if oo_resp.status_code == 200:
    oo_data = oo_resp.json()
    print("Open Opus — Beethoven search:\n")
    pp(oo_data)
else:
    print(f"Open Opus error: {oo_resp.status_code}")
    print(oo_resp.text[:500])

Open Opus — Beethoven search:

{
  "status": {
    "version": "1.20.6",
    "success": "true",
    "source": "db",
    "rows": 1,
    "processingtime": 0.0013680458068847656,
    "api": "Open Opus-dyn"
  },
  "request": {
    "type": "search",
    "item": "beethoven"
  },
  "composers": [
    {
      "id": "145",
      "name": "Beethoven",
      "complete_name": "Ludwig van Beethoven",
      "birth": "1770-01-01",
      "death": "1827-01-01",
      "epoch": "Early Romantic",
      "portrait": "https://assets.openopus.org/portraits/55910756-1568084860.jpg"
    }
  ]
}


In [11]:
# Get Beethoven's works by genre
# Open Opus genres: Orchestral, Chamber, Keyboard, Stage, Choral, Vocal

genres_to_test = ["Orchestral", "Chamber", "Keyboard"]

for genre in genres_to_test:
    time.sleep(1)
    resp = requests.get(
        f"https://api.openopus.org/work/list/composer/145/genre/{genre}/search/.json",
        headers=HEADERS
    )
    if resp.status_code == 200:
        data = resp.json()
        works = data.get("works", [])
        print(f"\n{genre} ({len(works)} works):")
        for w in works[:5]:
            print(f"  - {w.get('title', '?')} ({w.get('subtitle', '')})")
        if len(works) > 5:
            print(f"  ... and {len(works) - 5} more")
    else:
        print(f"  {genre}: error {resp.status_code}")


Orchestral (44 works):
  - Leonore Overture no. 3, op. 72b ()
  - Piano Concerto no. 3 in C minor, op. 37 ()
  - Piano Concerto no. 4 in G major, op. 58 ()
  - Piano Concerto no. 5 in E flat major, op. 73, "Emperor" ()
  - Symphony no. 3 in E flat major, op. 55, "Eroica" ()
  ... and 39 more

Chamber (108 works):
  - Cello Sonata no. 4 in C major, op. 102, no. 1 ()
  - Grosse Fuge, op. 133 (For string quartet)
  - Piano Trio no. 5 in D major, op. 70 no. 1, "Ghost" ()
  - Piano Trio no. 7 in B flat major, op. 97, "Archduke" ()
  - String Quartet no. 7 in F major, op. 59 no. 1, "Razumovsky" ()
  ... and 103 more

Keyboard (98 works):
  - Piano Sonata no. 8 in C minor, op. 13, "Pathétique" ()
  - Piano Sonata no. 17 in D minor, op. 31 no. 2, "Tempest" ()
  - Piano Sonata no. 21 in C major, op. 53, "Waldstein" ()
  - Piano Sonata no. 23 in F minor, op. 57, "Appassionata" ()
  - Piano Sonata no. 26 in E flat major, op. 81a, "Les Adieux" ()
  ... and 93 more


In [12]:
# Test: does Open Opus have non-classical artists?
for name in ["Miles Davis", "Fela Kuti", "Tom Jobim"]:
    time.sleep(1)
    resp = requests.get(
        f"https://api.openopus.org/composer/list/search/{name.replace(' ', '%20')}.json",
        headers=HEADERS
    )
    if resp.status_code == 200:
        data = resp.json()
        status = data.get("status", {})
        composers = data.get("composers", [])
        if composers:
            print(f"{name}: FOUND — {composers[0].get('complete_name', '?')}")
        else:
            print(f"{name}: NOT FOUND (classical only)")
    else:
        print(f"{name}: error {resp.status_code}")

Miles Davis: NOT FOUND (classical only)
Fela Kuti: NOT FOUND (classical only)
Tom Jobim: NOT FOUND (classical only)


## 4. MusicBrainz Baseline — Non-Western Artists

Check how well MusicBrainz covers Brazilian and African artists.

In [13]:
import musicbrainzngs
musicbrainzngs.set_useragent("KE-CW2-MusicHistory", "0.1", "lucas@example.com")

for name in ["Antônio Carlos Jobim", "Fela Kuti", "Miriam Makeba", "Youssou N'Dour"]:
    time.sleep(1.1)
    try:
        search = musicbrainzngs.search_artists(artist=name, limit=1)
        if search["artist-list"]:
            a = search["artist-list"][0]
            score = a.get("ext:score", "?")
            country = a.get("country", "?")
            atype = a.get("type", "?")
            print(f"{name:30s} → {a['name']} | Type: {atype} | Country: {country} | Score: {score}")
        else:
            print(f"{name:30s} → NOT FOUND")
    except Exception as e:
        print(f"{name:30s} → ERROR: {e}")

Antônio Carlos Jobim           → Antônio Carlos Jobim | Type: Person | Country: BR | Score: 100
Fela Kuti                      → Fela Kuti | Type: Person | Country: NG | Score: 100
Miriam Makeba                  → Miriam Makeba | Type: Person | Country: ZA | Score: 100
Youssou N'Dour                 → Youssou N’Dour | Type: Person | Country: SN | Score: 100


In [14]:
# Check tag/genre coverage for non-Western artists
for name, mbid in [("Fela Kuti", None), ("Antônio Carlos Jobim", None)]:
    time.sleep(1.1)
    search = musicbrainzngs.search_artists(artist=name, limit=1)
    if search["artist-list"]:
        mbid = search["artist-list"][0]["id"]
        time.sleep(1.1)
        detail = musicbrainzngs.get_artist_by_id(
            mbid, includes=["tags", "artist-rels", "url-rels"]
        )["artist"]
        
        tags = detail.get("tag-list", [])
        rels = detail.get("artist-relation-list", [])
        urls = detail.get("url-relation-list", [])
        
        # Find cross-reference links
        discogs = next((r["target"] for r in urls if r["type"] == "discogs"), "None")
        wikidata = next((r["target"] for r in urls if r["type"] == "wikidata"), "None")
        
        print(f"\n{detail['name']} ({mbid})")
        print(f"  Country: {detail.get('country', '?')}")
        print(f"  Tags: {[t['name'] for t in sorted(tags, key=lambda t: int(t.get('count', 0)), reverse=True)[:8]]}")
        print(f"  Artist rels: {len(rels)}")
        print(f"  Discogs: {discogs}")
        print(f"  Wikidata: {wikidata}")


Fela Kuti (6514cffa-fbe0-4965-ad88-e998ead8a82a)
  Country: NG
  Tags: ['afrobeat', 'african', 'nigerian', 'afro-beat', 'jazz', 'jùjú']
  Artist rels: 11
  Discogs: https://www.discogs.com/artist/19812
  Wikidata: https://www.wikidata.org/wiki/Q313868

Antônio Carlos Jobim (7a8dbe84-f4c0-4457-bfa3-edced1f8cde0)
  Country: BR
  Tags: ['bossa nova', 'latin jazz', 'jazz', 'latin', 'brazilian']
  Artist rels: 6
  Discogs: https://www.discogs.com/artist/27991
  Wikidata: https://www.wikidata.org/wiki/Q200131


## 5. Evaluation Summary

Fill in after running all cells.

### Source Comparison Matrix

| Feature | MusicBrainz | Discogs | Wikipedia | Wikidata | DBpedia | Open Opus |
|---|---|---|---|---|---|---|
| Data format | JSON | JSON | Text | RDF (SPARQL) | RDF (SPARQL) | JSON |
| Auth needed | No | No | No | No | No | No |
| Rate limit | 1/sec | 25/min | ~200/sec | Soft | Soft | Unknown |
| Rock/Pop coverage | | | | | | |
| Classical coverage | | | | | | |
| Jazz coverage | | | | | | |
| Brazilian coverage | | | | | | |
| African coverage | | | | | | |
| Genres | Tags (flat) | Genre+Style | Categories | P136 | genre | Classical genres |
| Awards | No | No | Text | P166 | Yes | No |
| Instruments | In rels | No | Text | P1303 | Yes | No |
| Influences | In rels | No | Text | P737 | Yes | No |
| Record labels | In releases | In releases | Text | P264 | Yes | No |
| Compositions | Works entity | No | Text | P86 (composer) | Yes | Yes (rich) |
| Cross-references | URL rels | URLs | Links | Sitelinks | owl:sameAs | No |

### Decision Matrix

| Source | Include? | Role | Justification |
|---|---|---|---|
| MusicBrainz | | | |
| Discogs | | | |
| Wikipedia | | | |
| Wikidata | | | |
| DBpedia | | | |
| Open Opus | | | |

## Next Steps

Based on the evaluation, finalise the data source selection and begin building the production pipeline.